In [1]:
from pathlib import Path
project_root = Path.cwd().parent
data_dir = project_root / "data" / "raw" / "isot"

print("Project root:", project_root)
print("Dataset folder:", data_dir)
print("\nFiles found:")

for file in data_dir.iterdir():
    print(file.name, "-", round(file.stat().st_size / (1024 * 1024), 2), "MB")

Project root: c:\Users\KIIT\clickbait-credibility-system
Dataset folder: c:\Users\KIIT\clickbait-credibility-system\data\raw\isot

Files found:
Fake.csv - 59.88 MB
True.csv - 51.1 MB


In [2]:
import pandas as pd
fake_path = data_dir / "Fake.csv"
true_path = data_dir / "True.csv"
fake_df = pd.read_csv(fake_path)
true_df = pd.read_csv(true_path)
print("Fake dataset shape:", fake_df.shape)
print("Real dataset shape:", true_df.shape)

print("\nFake dataset columns:")
print(fake_df.columns.tolist())

print("\nReal dataset columns:")
print(true_df.columns.tolist())

print("\nFirst 3 fake articles:")
display(fake_df.head(3))

print("\nFirst 3 real articles:")
display(true_df.head(3))

Fake dataset shape: (23481, 4)
Real dataset shape: (21417, 4)

Fake dataset columns:
['title', 'text', 'subject', 'date']

Real dataset columns:
['title', 'text', 'subject', 'date']

First 3 fake articles:


,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"



First 3 real articles:


,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"


In [3]:
import re
import pandas as pd
# Work on copies so the original loaded data remains unchanged.
fake = fake_df.copy()
real = true_df.copy()
# Standardize column names and remove accidental surrounding whitespace.
fake.columns = fake.columns.str.strip().str.lower()
real.columns = real.columns.str.strip().str.lower()
# Check missing values and exact duplicate rows.
print("=== DATA QUALITY AUDIT ===")

for name, df in [("Fake", fake), ("Real", real)]:
    print(f"\n{name} articles: {len(df):,}")
    print("Missing values:")
    print(df.isna().sum().to_dict())
    print("Exact duplicate rows:", int(df.duplicated().sum()))
# Normalize titles for comparison only; preserve original titles for now.
def normalize_title(series):
    return (
        series.astype("string")
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
fake["_title_key"] = normalize_title(fake["title"])
real["_title_key"] = normalize_title(real["title"])

fake_keys = set(fake["_title_key"].dropna())
real_keys = set(real["_title_key"].dropna())
print("\n=== TITLE AUDIT ===")
print("Fake rows with missing titles:", int(fake["title"].isna().sum()))
print("Real rows with missing titles:", int(real["title"].isna().sum()))
print("Fake duplicate titles:", int(fake["_title_key"].duplicated().sum()))
print("Real duplicate titles:", int(real["_title_key"].duplicated().sum()))
print("Titles appearing in BOTH classes:", len(fake_keys & real_keys))

print("\n=== SUBJECT DISTRIBUTION ===")
print("\nFake articles:")
print(fake["subject"].value_counts(dropna=False).head(10))
print("\nReal articles:")
print(real["subject"].value_counts(dropna=False).head(10))

# Show examples of titles that appear in both classes, if any.
overlap = fake_keys & real_keys
if overlap:
    print("\nExamples of cross-class title overlap:")
    print(list(overlap)[:10])

=== DATA QUALITY AUDIT ===

Fake articles: 23,481
Missing values:
{'title': 0, 'text': 0, 'subject': 0, 'date': 0}
Exact duplicate rows: 3

Real articles: 21,417
Missing values:
{'title': 0, 'text': 0, 'subject': 0, 'date': 0}
Exact duplicate rows: 206

=== TITLE AUDIT ===
Fake rows with missing titles: 0
Real rows with missing titles: 0
Fake duplicate titles: 5580
Real duplicate titles: 593
Titles appearing in BOTH classes: 0

=== SUBJECT DISTRIBUTION ===

Fake articles:
subject
News               9050
politics           6841
left-news          4459
Government News    1570
US_News             783
Middle-east         778
Name: count, dtype: int64

Real articles:
subject
politicsNews    11272
worldnews       10145
Name: count, dtype: int64


In [4]:
import re
import pandas as pd
# Inspect representative real and fake articles.
print("=== REAL ARTICLE EXAMPLES ===")
for text in true_df["text"].dropna().head(5):
    print(repr(str(text)[:250]))
    print()
print("=== FAKE ARTICLE EXAMPLES ===")
for text in fake_df["text"].dropna().head(5):
    print(repr(str(text)[:250]))
    print()

=== REAL ARTICLE EXAMPLES ===
'WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal conservative” on Sunday and urged budget restraint'

'WASHINGTON (Reuters) - Transgender people will be allowed for the first time to enlist in the U.S. military starting on Monday as ordered by federal courts, the Pentagon said on Friday, after President Donald Trump’s administration decided not to app'

'WASHINGTON (Reuters) - The special counsel investigation of links between Russia and President Trump’s 2016 election campaign should continue without interference in 2018, despite calls from some Trump administration allies and Republican lawmakers t'

'WASHINGTON (Reuters) - Trump campaign adviser George Papadopoulos told an Australian diplomat in May 2016 that Russia had political dirt on Democratic presidential candidate Hillary Clinton, the New York Tim

In [5]:
# Count leading Reuters-style prefixes in the original article text.
reuters_pattern = re.compile(
    r"^\s*[^()\n]{1,100}\s*\(Reuters\)\s*[-–—:]\s*",
    flags=re.IGNORECASE
)
def has_reuters_prefix(text):
    if pd.isna(text):
        return False
    return bool(reuters_pattern.search(str(text)))
real_reuters_rate = true_df["text"].map(has_reuters_prefix).mean()
fake_reuters_rate = fake_df["text"].map(has_reuters_prefix).mean()
print(f"Real articles with Reuters prefix: {real_reuters_rate:.2%}")
print(f"Fake articles with Reuters prefix: {fake_reuters_rate:.2%}")

Real articles with Reuters prefix: 91.58%
Fake articles with Reuters prefix: 0.00%


In [6]:
import re
import pandas as pd
# Match only a clear Reuters dateline at the start of the article.
reuters_prefix = re.compile(
    r"^\s*[A-Z][A-Z\s,./'-]{0,100}\s+\(Reuters\)\s*[-–—:]+\s*"
)
def remove_reuters_prefix(text):
    if pd.isna(text):
        return ""
    return reuters_prefix.sub("", str(text), count=1).strip()
# Rebuild from original data every time.
true_clean = true_df.copy()
fake_clean = fake_df.copy()
true_clean["text_clean"] = true_clean["text"].map(remove_reuters_prefix)
fake_clean["text_clean"] = fake_clean["text"].map(remove_reuters_prefix)
print("Real rows changed:",(true_clean["text"] != true_clean["text_clean"]).sum())
print("Fake rows changed:",(fake_clean["text"] != fake_clean["text_clean"]).sum())

Real rows changed: 21296
Fake rows changed: 6302


In [7]:
# Check whether any fake text changed without a Reuters-prefix match.
changed_fake = fake_clean.loc[
    fake_clean["text"] != fake_clean["text_clean"],
    ["title", "text", "text_clean"]
]
unmatched_changes = []
for _, row in changed_fake.iterrows():
    if reuters_prefix.match(str(row["text"])) is None:
        unmatched_changes.append(row)
print("Fake rows changed:", len(changed_fake))
print("Changes without a matching Reuters prefix:", len(unmatched_changes))
if unmatched_changes:
    print("\nFirst unmatched change:")
    row = unmatched_changes[0]
    print("TITLE:", row["title"])
    print("BEFORE:", repr(str(row["text"])[:200]))
    print("AFTER:", repr(str(row["text_clean"])[:200]))

Fake rows changed: 6302
Changes without a matching Reuters prefix: 6302

First unmatched change:
TITLE:  WATCH: Fox Host Calls For A ‘Cleansing’ Of The FBI, And To Arrest Everyone Investigating Trump
BEFORE: ' Judge  Jeanine Pirro has continued her screamy ragey meltdown over special counsel Robert Mueller s investigation into any possible collusion between the Trump campaign and Russia during the 2016 ele'
AFTER: 'Judge  Jeanine Pirro has continued her screamy ragey meltdown over special counsel Robert Mueller s investigation into any possible collusion between the Trump campaign and Russia during the 2016 elec'


In [8]:
# If this is True, the ONLY difference between text and text_clean 
# for unmatched rows is whitespace — confirming no content was altered
only_whitespace_diff = (
    fake_clean["text"].astype(str).str.strip() == fake_clean["text_clean"]
).all()
print("Fake articles: only whitespace trimmed (no content corruption)?", only_whitespace_diff)
# Also confirm real articles truly got their Reuters prefix stripped
print("\nSample real article before/after:")
sample_real = true_clean[true_clean["text"] != true_clean["text_clean"]].iloc[0]
print("BEFORE:", repr(str(sample_real["text"])[:150]))
print("AFTER:", repr(str(sample_real["text_clean"])[:150]))

Fake articles: only whitespace trimmed (no content corruption)? True

Sample real article before/after:
BEFORE: 'WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national d'
AFTER: 'The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts'


In [9]:
import pandas as pd
from pathlib import Path
# Keep the cleaned article text and assign labels. replaced
# FINAL CREDIBILITY DATASET 0 = Fake, 1 = Real
# Build the real-article dataset.
real_data = pd.DataFrame({
    "headline": true_clean["title"].fillna("").astype(str),
    "text": true_clean["text_clean"].fillna("").astype(str),
    "label": 1
})
# Build the fake-article dataset.
fake_data = pd.DataFrame({
    "headline": fake_clean["title"].fillna("").astype(str),
    "text": fake_clean["text_clean"].fillna("").astype(str),
    "label": 0
})

# Combine both classes.
credibility_df = pd.concat(
    [real_data, fake_data],
    ignore_index=True
)

# Normalize whitespace without modifying the meaning of the text.
for column in ["headline", "text"]:
    credibility_df[column] = (
        credibility_df[column]
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# Remove empty headlines.
credibility_df = credibility_df[
    credibility_df["headline"].ne("")
].copy()

# Create normalized headline keys for duplicate detection.
credibility_df["_headline_key"] = (
    credibility_df["headline"]
    .str.lower()
    .str.replace(r"[^a-z0-9]+", " ", regex=True)
    .str.strip()
)

# Find normalized headlines that have conflicting labels.
labels_per_headline = (
    credibility_df
    .groupby("_headline_key")["label"]
    .nunique()
)

conflicting_keys = labels_per_headline[
    labels_per_headline > 1
].index

conflicting_rows = credibility_df[
    credibility_df["_headline_key"].isin(conflicting_keys)
]

print("Rows with conflicting headline labels:", len(conflicting_rows))

# Remove all rows belonging to conflicting headline groups.
credibility_df = credibility_df[
    ~credibility_df["_headline_key"].isin(conflicting_keys)
].copy()

# Keep one record per normalized headline.
credibility_df = credibility_df.drop_duplicates(
    subset="_headline_key",
    keep="first"
).copy()

# Remove the temporary helper column.
credibility_df = credibility_df.drop(
    columns=["_headline_key"]
).reset_index(drop=True)
# FINAL DATASET VALIDATION

print("\nFinal dataset shape:", credibility_df.shape)
print("\nColumns:", credibility_df.columns.tolist())

print("\nClass counts:")
print(
    credibility_df["label"]
    .value_counts()
    .sort_index()
    .rename(index={0: "Fake", 1: "Real"})
)

print("\nClass proportions:")
print(
    credibility_df["label"]
    .value_counts(normalize=True)
    .sort_index()
    .rename(index={0: "Fake", 1: "Real"})
    .round(4)
)

print("\nMissing values:")
print(credibility_df.isna().sum())

print("\nEmpty headlines:", credibility_df["headline"].str.strip().eq("").sum())

normalized_headlines = (
    credibility_df["headline"]
    .str.lower()
    .str.replace(r"[^a-z0-9]+", " ", regex=True)
    .str.strip()
)

print(
    "\nDuplicate normalized headlines remaining:",
    normalized_headlines.duplicated().sum()
)

print("\nExample cleaned headline:")
print(credibility_df["headline"].iloc[0])

print("\nExample cleaned article text:")
print(credibility_df["text"].iloc[0][:500])

Rows with conflicting headline labels: 0

Final dataset shape: (38720, 3)

Columns: ['headline', 'text', 'label']

Class counts:
label
Fake    17901
Real    20819
Name: count, dtype: int64

Class proportions:
label
Fake    0.4623
Real    0.5377
Name: proportion, dtype: float64

Missing values:
headline    0
text        0
label       0
dtype: int64

Empty headlines: 0

Duplicate normalized headlines remaining: 0

Example cleaned headline:
As U.S. budget fight looms, Republicans flip their fiscal script

Example cleaned article text:
The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal conservative” on Sunday and urged budget restraint in 2018. In keeping with a sharp pivot under way among Republicans, U.S. Representative Mark Meadows, speaking on CBS’ “Face the Nation,” drew a hard line on federal spending, which lawmakers are bracing to do battle over in January. 

In [10]:
from pathlib import Path
project_root = Path.cwd().parent
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
output_path = processed_dir / "credibility_clean.csv"
credibility_df.to_csv(output_path, index=False)
print("Saved dataset:", output_path)
print("Number of rows:", len(credibility_df))

Saved dataset: c:\Users\KIIT\clickbait-credibility-system\data\processed\credibility_clean.csv
Number of rows: 38720


In [11]:
from sklearn.model_selection import train_test_split
# Use the correct combined dataset.
df_model = credibility_df.copy()
# Confirm required columns exist.
print("Dataset shape:", df_model.shape)
print("Columns:", df_model.columns.tolist())
# Inputs and target.
X = df_model["headline"].astype(str)
y = df_model["label"].astype(int)
# 80% training, 10% validation, 10% testing.
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)
print("\nTraining samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))
print("\nTraining label counts:")
print(y_train.value_counts().sort_index())
print("\nTraining label proportions:")
print(y_train.value_counts(normalize=True).sort_index())

Dataset shape: (38720, 3)
Columns: ['headline', 'text', 'label']

Training samples: 30976
Validation samples: 3872
Test samples: 3872

Training label counts:
label
0    14321
1    16655
Name: count, dtype: int64

Training label proportions:
label
0    0.462326
1    0.537674
Name: proportion, dtype: float64


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
# Convert headlines into numerical features.
# Fit only on training data to avoid data leakage.
tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    max_features=100_000
)
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

print("Training matrix:", X_train_tfidf.shape)
print("Validation matrix:", X_val_tfidf.shape)
print("Test matrix:", X_test_tfidf.shape)

Training matrix: (30976, 51223)
Validation matrix: (3872, 51223)
Test matrix: (3872, 51223)


In [13]:
from sklearn.linear_model import LogisticRegression
credibility_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)
credibility_model.fit(X_train_tfidf, y_train)
print("Credibility model training completed.")

Credibility model training completed.


In [14]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)
y_val_pred = credibility_model.predict(X_val_tfidf)
y_val_prob = credibility_model.predict_proba(X_val_tfidf)[:, 1]
print("Validation accuracy:", round(accuracy_score(y_val, y_val_pred), 4))
print("\nClassification report:")
print(
    classification_report(
        y_val,
        y_val_pred,
        target_names=["Fake (0)", "Real (1)"],
        zero_division=0
    )
)
print("Confusion matrix:")
print(confusion_matrix(y_val, y_val_pred))
print(
    "Validation ROC-AUC:",
    round(roc_auc_score(y_val, y_val_prob), 4)
)

Validation accuracy: 0.9473

Classification report:
              precision    recall  f1-score   support

    Fake (0)       0.94      0.95      0.94      1790
    Real (1)       0.95      0.95      0.95      2082

    accuracy                           0.95      3872
   macro avg       0.95      0.95      0.95      3872
weighted avg       0.95      0.95      0.95      3872

Confusion matrix:
[[1693   97]
 [ 107 1975]]
Validation ROC-AUC: 0.9882


In [15]:
feature_names = tfidf.get_feature_names_out()
coefs = credibility_model.coef_[0]
feature_coef_pairs = list(zip(feature_names, coefs))
# Sort by coefficient value, descending = strongest REAL indicators first
top_real_indicators = sorted(feature_coef_pairs, key=lambda x: x[1], reverse=True)[:20]
# Sort by coefficient value, ascending = strongest FAKE indicators first
top_fake_indicators = sorted(feature_coef_pairs, key=lambda x: x[1])[:20]
print("Top words/n-grams pushing toward REAL:")
for word, coef in top_real_indicators:
    print(f"  {word}: {coef:.3f}")
print("\nTop words/n-grams pushing toward FAKE:")
for word, coef in top_fake_indicators:
    print(f"  {word}: {coef:.3f}")

Top words/n-grams pushing toward REAL:
  says: 8.858
  factbox: 5.101
  china: 4.596
  house: 3.975
  eu: 3.929
  tax: 3.490
  talks: 3.436
  senate: 3.403
  trump says: 3.258
  pm: 3.199
  court: 3.113
  turkey: 3.040
  myanmar: 3.031
  brexit: 3.018
  lawmakers: 2.938
  urges: 2.914
  minister: 2.845
  britain: 2.840
  north: 2.794
  russia: 2.762

Top words/n-grams pushing toward FAKE:
  video: -20.227
  the: -10.121
  watch: -8.922
  hillary: -8.487
  breaking: -8.173
  gop: -7.763
  just: -7.463
  is: -6.781
  and: -6.351
  us: -5.830
  his: -5.327
  tweets: -5.049
  you: -4.919
  this: -4.878
  he: -4.758
  details: -4.470
  her: -4.366
  black: -4.318
  are: -4.288
  racist: -4.268


In [16]:
# CREDIBILITY MODEL: HEADLINE + ARTICLE TEXT
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)
# 1. Verify that the required variables exist
required_variables = [
    "df_model",
    "X_train", "X_val", "X_test",
    "y_train", "y_val", "y_test"
]
missing_variables = [
    name for name in required_variables
    if name not in globals()
]
if missing_variables:
    raise NameError(
        f"Missing variables: {missing_variables}. "
        "Run the preceding dataset and split cells first."
    )
# 2. Verify the dataset columns
required_columns = {"headline", "text", "label"}
if not required_columns.issubset(df_model.columns):
    raise ValueError(
        f"df_model must contain {required_columns}. "
        f"Found: {list(df_model.columns)}"
    )

# 3. Check that the split indexes match the dataset
for split_name, split_X, split_y in [
    ("Training", X_train, y_train),
    ("Validation", X_val, y_val),
    ("Test", X_test, y_test)
]:
    if not split_X.index.equals(split_y.index):
        raise ValueError(
            f"{split_name} feature and label indexes do not match."
        )
    if not split_X.index.isin(df_model.index).all():
        raise ValueError(
            f"{split_name} indexes do not match df_model."
        )
print("Dataset and split checks passed.")
# 4. Combine headline and cleaned article text
combined_text = (
    df_model["headline"].fillna("").astype(str).str.strip()
    + " "
    + df_model["text"].fillna("").astype(str).str.strip()
).str.strip()
# 5. Create text for each existing split
X_train_full = combined_text.loc[X_train.index]
X_val_full = combined_text.loc[X_val.index]
X_test_full = combined_text.loc[X_test.index]
# 6. Create and fit TF-IDF using training data only
tfidf_full = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    max_features=100_000
)

X_train_full_tfidf = tfidf_full.fit_transform(X_train_full)
X_val_full_tfidf = tfidf_full.transform(X_val_full)

print("Training matrix shape:", X_train_full_tfidf.shape)
print("Validation matrix shape:", X_val_full_tfidf.shape)

# 7. Train the credibility model
credibility_full_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)
credibility_full_model.fit(X_train_full_tfidf, y_train)

print("\nFull-text credibility model trained successfully.")

# 8. Evaluate on the validation set
y_val_full_pred = credibility_full_model.predict(
    X_val_full_tfidf
)

y_val_full_prob = credibility_full_model.predict_proba(
    X_val_full_tfidf
)[:, 1]

print("\n========== VALIDATION RESULTS ==========")

print(
    "Accuracy:",
    round(accuracy_score(y_val, y_val_full_pred), 4)
)

print(
    "ROC-AUC:",
    round(roc_auc_score(y_val, y_val_full_prob), 4)
)
print("\nClassification Report:")
print(
    classification_report(
        y_val,
        y_val_full_pred,
        labels=[0, 1],
        target_names=["Fake", "Real"],
        digits=4
    )
)
print("Confusion Matrix:")
print(
    confusion_matrix(
        y_val,
        y_val_full_pred,
        labels=[0, 1]
    )
)

Dataset and split checks passed.
Training matrix shape: (30976, 100000)
Validation matrix shape: (3872, 100000)

Full-text credibility model trained successfully.

========== VALIDATION RESULTS ==========
Accuracy: 0.9892
ROC-AUC: 0.9989

Classification Report:
              precision    recall  f1-score   support

        Fake     0.9905    0.9860    0.9882      1790
        Real     0.9880    0.9918    0.9899      2082

    accuracy                         0.9892      3872
   macro avg     0.9892    0.9889    0.9891      3872
weighted avg     0.9892    0.9892    0.9892      3872

Confusion Matrix:
[[1765   25]
 [  17 2065]]


In [17]:
# FINAL TEST EVALUATION: FULL-TEXT MODEL
from sklearn.metrics import ( accuracy_score, classification_report, confusion_matrix, roc_auc_score)
# Transform test data using the already-fitted TF-IDF vectorizer
X_test_full_tfidf = tfidf_full.transform(X_test_full)

# Predict labels and probabilities
y_test_full_pred = credibility_full_model.predict(
    X_test_full_tfidf
)

y_test_full_prob = credibility_full_model.predict_proba(
    X_test_full_tfidf
)[:, 1]

# Report final test metrics
print("========== FINAL TEST RESULTS ==========")
print(
    "Accuracy:",
    round(accuracy_score(y_test, y_test_full_pred), 4)
)

print(
    "ROC-AUC:",
    round(roc_auc_score(y_test, y_test_full_prob), 4)
)
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_test_full_pred,
        labels=[0, 1],
        target_names=["Fake", "Real"],
        digits=4
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_test_full_pred,
        labels=[0, 1]
    )
)

========== FINAL TEST RESULTS ==========
Accuracy: 0.9855
ROC-AUC: 0.9979

Classification Report:
              precision    recall  f1-score   support

        Fake     0.9887    0.9799    0.9843      1790
        Real     0.9828    0.9904    0.9866      2082

    accuracy                         0.9855      3872
   macro avg     0.9858    0.9851    0.9854      3872
weighted avg     0.9856    0.9855    0.9855      3872

Confusion Matrix:
[[1754   36]
 [  20 2062]]


In [18]:
# SAVE THE FULL-TEXT CREDIBILITY MODEL
import joblib
from pathlib import Path
# Project root: parent of the notebooks directory
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save the trained model and fitted vectorizer
joblib.dump(
    credibility_full_model,
    MODEL_DIR / "credibility_model.joblib"
)

joblib.dump(
    tfidf_full,
    MODEL_DIR / "credibility_tfidf.joblib"
)
print("Credibility model saved successfully.")
print("Model:", MODEL_DIR / "credibility_model.joblib")
print("Vectorizer:", MODEL_DIR / "credibility_tfidf.joblib")

# Verify that both files exist
assert (MODEL_DIR / "credibility_model.joblib").exists()
assert (MODEL_DIR / "credibility_tfidf.joblib").exists()

print("Both saved files verified.")

Credibility model saved successfully.
Model: c:\Users\KIIT\clickbait-credibility-system\models\credibility_model.joblib
Vectorizer: c:\Users\KIIT\clickbait-credibility-system\models\credibility_tfidf.joblib
Both saved files verified.


In [19]:
# VERIFY SAVED CREDIBILITY MODEL
import joblib
from pathlib import Path
# Use the same model directory as the previous cell
model_path = MODEL_DIR / "credibility_model.joblib"
vectorizer_path = MODEL_DIR / "credibility_tfidf.joblib"
# Load the saved artifacts
loaded_credibility_model = joblib.load(model_path)
loaded_credibility_tfidf = joblib.load(vectorizer_path)

# Example headline and article text
sample_headline = (
    "Government announces new economic policy"
)
sample_article = (
    "The government announced a new economic policy today. "
    "Officials said the policy would be reviewed by the relevant "
    "departments before implementation."
)
# Combine the inputs in the same format used during training
sample_text = sample_headline + " " + sample_article
# Transform and predict
sample_vector = loaded_credibility_tfidf.transform([sample_text])
predicted_label = loaded_credibility_model.predict(sample_vector)[0]
probabilities = loaded_credibility_model.predict_proba(sample_vector)[0]
# Map class labels to readable names
predicted_class = "Real" if predicted_label == 1 else "Fake"
print("Saved model loaded successfully.")
print("Predicted class:", predicted_class)
print("Probability of Fake:", round(probabilities[0], 4))
print("Probability of Real:", round(probabilities[1], 4))

Saved model loaded successfully.
Predicted class: Real
Probability of Fake: 0.2111
Probability of Real: 0.7889


In [1]:
import sys
from pathlib import Path
# Ensure the project root is importable
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.credibility_predictor import predict_credibility

result = predict_credibility(
    headline="Government announces new economic policy",
    article_text=(
        "The government announced a new economic policy today. "
        "Officials said the policy would be reviewed by the relevant "
        "departments before implementation."
    ),
)

print(result)

{'predicted_class': 'Real', 'fake_probability': 0.2111, 'real_probability': 0.7889}
